In [1]:
# --- Import BeautifulSoup to web scrape and XML parser to read site map! Import Pandas to aid with analysis!---

# ~ This is BeautifulSoup imports ~
from bs4 import BeautifulSoup
import requests

# ~ This is XML parser imports ~
from parsel import Selector
import httpx

# ~ This is Pandas import ~
import pandas as pd

In [2]:
# --- Let's examine the XML doc! ---

# ~ Sally's needs 2 site maps to find all of her recipes ~
sitemap1 = 'https://sallysbakingaddiction.com/post-sitemap.xml'
sitemap2 = 'https://sallysbakingaddiction.com/post-sitemap2.xml'

page1 = httpx.get(sitemap1)
page2 = httpx.get(sitemap2)

soup1 = BeautifulSoup(page1.text, 'xml')
soup2 = BeautifulSoup(page2.text, 'xml')

# ~ To get an idea of what we're dealing with ~
print(soup1.prettify())

<?xml version="1.0" encoding="utf-8"?>
<?xml-stylesheet type="text/xsl" href="//sallysbakingaddiction.com/wp-content/plugins/wordpress-seo/css/main-sitemap.xsl"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9" xmlns:image="http://www.google.com/schemas/sitemap-image/1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.sitemaps.org/schemas/sitemap/0.9 http://www.sitemaps.org/schemas/sitemap/0.9/sitemap.xsd http://www.google.com/schemas/sitemap-image/1.1 http://www.google.com/schemas/sitemap-image/1.1/sitemap-image.xsd">
 <url>
  <loc>
   https://sallysbakingaddiction.com/recent-posts/
  </loc>
  <lastmod>
   2026-06-16T16:01:54+00:00
  </lastmod>
 </url>
 <url>
  <loc>
   https://sallysbakingaddiction.com/nutella-frosting/
  </loc>
  <lastmod>
   2020-08-24T22:06:46+00:00
  </lastmod>
  <image:image>
   <image:loc>
    https://sallysbakingaddiction.com/wp-content/uploads/2019/01/nutella-frosting.jpg
   </image:loc>
  </image:image>
  <im

In [3]:
# --- Puts the recipes in a list + data frame! Will make it easier to use these links! ---

# ~ Fetching the urls ~
list1 = []
list2 = []

list1 = soup1.find_all('loc') # Will give us all the urls to recipes AND images so need to filter :(
list2 = soup2.find_all('loc')
del list1[0] # Gets rid of non-recipe url
recipe = []

# ~ Troubleshooted thanks to StackOverflow but to understand logic: ~
    #For every url given from the soup, we'll go through each character in the url to detect the '.jpg'. 
    #Want to use .split() to ensure we filter specifically for it.
    #If '.jpg' was detected - yay! We will now use the not command to ensure it will filter out all of the image urls.
    #We can now safely add the recipe urls to the recipe list!

for i in list1: 
    if not(any(j in str(i) for j in 'jpg'.split())): 
        recipe.append(i.text)

for i in list2: 
    if not(any(j in str(i) for j in 'jpg'.split())): 
        recipe.append(i.text)

urldf = pd.DataFrame(recipe)

# ~ Extracting the recipe titles from the url ~
name = []

for i in recipe:
    title = i.split("/")[-2] # Titles start after the 2nd last /!
    title = title.replace('-', ' ').title()
    name.append(title)
    
namedf = pd.DataFrame(name)

In [4]:
# --- Now we'll extract the data we need for our personal ranking system! ---

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

average = []
count = []
comment = []

# ~ To have one session for the loop to reuse rather than opening multiple ~
session = requests.Session()
session.headers.update(headers)

#Need to figure out how to better optimize so it runs faster! Super slow currently...
for url in recipe:
    page = session.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    
    avg = soup.find(class_ = 'average')
    average.append(avg.text.strip() if avg else None)

    cnt = soup.find(class_ = 'count')
    count.append(cnt.text.strip() if cnt else None)

    com = soup.find(class_ = 'jump-button-link comment-jump')
    comment.append(com.text.strip().split()[0].replace(',', '') if com else '0')

In [5]:
# --- Make this into one dataframe with all our needed data! ---

# ~ Turning the lists to dataframes here since last cell took so long ~
# Can probably optimize better and have it added to the for loop in the last cell
averagedf = pd.DataFrame(average)
countdf = pd.DataFrame(count)
commentdf = pd.DataFrame(comment)

# ~ One big, happy dataframe! :D ~
recipedf = pd.concat([namedf, averagedf, countdf, commentdf, urldf], axis = 1, join = 'outer')
recipedf.columns = ['Title', 'Rating', 'Rating Count', 'Comment Count', 'URL']

# ~ Clean up the data to only include recipes we need ~
pd.set_option('display.max.rows', 20)
recipedf = recipedf.dropna()

for i in recipedf.index:
    if 'How To' in recipedf.loc[i, 'Title']: # Removes the How To's tutorials
        recipedf.drop(i, inplace = True)

# ~ Let's fix our Rating Count and Comment Count so it properly sorts
recipedf['Rating Count'] = recipedf['Rating Count'].astype(int)
recipedf['Comment Count'] = recipedf['Comment Count'].astype(int)

#recipedf.sort_values(by = 'Comment Count', ascending = True)

,Title,Rating,Rating Count,Comment Count,URL
606,Dark Chocolate Almond Butter Cookies,5,4,0,https://sallysbakingaddiction.com/dark-chocola...
378,Take 2 Funfetti Cake Batter Blondies,4.6,13,0,https://sallysbakingaddiction.com/take-2-funfe...
175,Glazed Cranberry Orange Scones,4.8,175,0,https://sallysbakingaddiction.com/glazed-cranb...
377,Nutella Brownies,4.9,117,0,https://sallysbakingaddiction.com/nutella-brow...
376,Peanut Butter Fudge,4.9,27,0,https://sallysbakingaddiction.com/peanut-butte...
...,...,...,...,...,...
424,Sandwich Bread,4.9,1144,3392,https://sallysbakingaddiction.com/sandwich-bread/
604,Banana Muffins,4.9,2168,4061,https://sallysbakingaddiction.com/banana-muffins/
1126,Best Banana Bread Recipe,4.8,1756,4164,https://sallysbakingaddiction.com/best-banana-...
1171,Triple Chocolate Layer Cake,4.9,1370,4954,https://sallysbakingaddiction.com/triple-choco...


In [6]:
# --- Making a ranking system to sort through the recipes ---

# ~ Logic of ranking system: ~
# Most popular recipes won't have the perfect rating, but rather it's been tested many times
# If the recipe has a high rating from many reviewers, we can safely assume this recipe is well loved and popular
# With recipes with very similar ratings and rating counts, we'll refer to the comment count
# Recipes with a larger comment count reaffirms the reliability and use of it,
# so much so people will write comments most likely figuring out ways to adjust the recipe to fit their dietary needs or pantry availability
recipedf.sort_values(by = ['Rating Count', 'Rating', 'Comment Count'], ascending = False)

# ~ Sally's Baking Addiction's Top 10 Recipes! ~
top10 = pd.DataFrame(recipedf.sort_values(by = ['Rating Count', 'Rating', 'Comment Count'], ascending = False))
top10 = top10.reset_index(drop = True)
top10 = top10.head(n = 10)
top10

,Title,Rating,Rating Count,Comment Count,URL
0,Banana Muffins,4.9,2168,4061,https://sallysbakingaddiction.com/banana-muffins/
1,Chewy Chocolate Chip Cookies,4.7,1952,6344,https://sallysbakingaddiction.com/chewy-chocol...
2,Best Banana Bread Recipe,4.8,1756,4164,https://sallysbakingaddiction.com/best-banana-...
3,Whole Wheat Bread,4.9,1589,3038,https://sallysbakingaddiction.com/whole-wheat-...
4,Triple Chocolate Layer Cake,4.9,1370,4954,https://sallysbakingaddiction.com/triple-choco...
5,Lemon Bars Recipe,4.7,1298,3158,https://sallysbakingaddiction.com/lemon-bars-r...
6,Sandwich Bread,4.9,1144,3392,https://sallysbakingaddiction.com/sandwich-bread/
7,Homemade Artisan Bread,4.8,1137,3219,https://sallysbakingaddiction.com/homemade-art...
8,Soft Chewy Oatmeal Raisin Cookies,4.8,1081,2821,https://sallysbakingaddiction.com/soft-chewy-o...
9,Soft Dinner Rolls,4.8,1064,2865,https://sallysbakingaddiction.com/soft-dinner-...


In [232]:
# --- Create the Ingredients Column! ---

pd.set_option('display.max_colwidth', None)

# ~ Get all the ingredients used in each recipe ~
ings = []
for i in top10.index:
    url = top10.loc[i, 'URL']
    page = session.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    body = soup.find(class_ = 'tasty-recipes-ingredients-body')
    
    ing = body.find_all('strong')
    for j in ing:
        if 'optional' in j.text.strip():
            continue
        if j.text.strip().title() in ings: # To remove duplicate ingredients inside a recipe
            continue
        ings.append(str(j.text.strip()).title())
    top10.loc[i, 'Ingredients'] = ', '.join(ings)
    ings = []

# ~ Clean up the data ~
replacements = [ # Created list to make it easier to replace string
    ('*', ''),
    ("’", ''),
    (',,', ','),
    ('Coarse Salt', 'Salt'),
    ('Egg Yolk, ', ''),
    ('Egg', 'Eggs'),
    ('Eggss', 'Eggs'),
    ('Instant ', ''),
    ('Active Dry Or ', ''),
    ('Whole ', ''),
    (', Chocolate Chunks', ''),
    ('Fresh ', '')
]
for old, new in replacements:
    top10['Ingredients'] = top10['Ingredients'].str.replace(old, new, regex = False)

top10

,Title,Rating,Rating Count,Comment Count,URL,Ingredients
0,Banana Muffins,4.9,2168,4061,https://sallysbakingaddiction.com/banana-muffins/,"All-Purpose Flour, Baking Powder, Baking Soda, Salt, Ground Cinnamon, Ground Nutmeg, Mashed Bananas, Unsalted Butter, Brown Sugar, Eggs, Pure Vanilla Extract, Milk"
1,Chewy Chocolate Chip Cookies,4.7,1952,6344,https://sallysbakingaddiction.com/chewy-chocolate-chip-cookies/,"All-Purpose Flour, Baking Soda, Cornstarch, Salt, Unsalted Butter, Brown Sugar, Granulated Sugar, Eggs, Pure Vanilla Extract, Semi-Sweet Chocolate Chips"
2,Best Banana Bread Recipe,4.8,1756,4164,https://sallysbakingaddiction.com/best-banana-bread-recipe/,"All-Purpose Flour, Baking Soda, Salt, Ground Cinnamon, Unsalted Butter, Brown Sugar, Eggs, Mashed Bananas, Plain Greek Yogurt, Sour Cream, Pure Vanilla Extract"
3,Whole Wheat Bread,4.9,1589,3038,https://sallysbakingaddiction.com/whole-wheat-bread/,"Water, Milk, Yeast, Wheat Flour, Honey, Unsalted Butter, Lemon Juice, Salt"
4,Triple Chocolate Layer Cake,4.9,1370,4954,https://sallysbakingaddiction.com/triple-chocolate-layer-cake/,"All-Purpose Flour, Cocoa Powder, Granulated Sugar, Baking Soda, Baking Powder, Salt, Espresso Powder, Vegetable Oil, Eggs, Pure Vanilla Extract, Buttermilk, Hot Coffee, Unsalted Butter, Confectioners Sugar, Unsweetened Cocoa Powder, Heavy Cream"
5,Lemon Bars Recipe,4.7,1298,3158,https://sallysbakingaddiction.com/lemon-bars-recipe/,"Unsalted Butter, Granulated Sugar, Pure Vanilla Extract, Salt, All-Purpose Flour, Eggs, Lemon Juice, Lemon Zest"
6,Sandwich Bread,4.9,1144,3392,https://sallysbakingaddiction.com/sandwich-bread/,"Water, Milk, Yeast, Granulated Sugar, Unsalted Butter, Salt, All-Purpose Flour, Bread Flour"
7,Homemade Artisan Bread,4.8,1137,3219,https://sallysbakingaddiction.com/homemade-artisan-bread/,"Bread Flour, Yeast, Salt, Water"
8,Soft Chewy Oatmeal Raisin Cookies,4.8,1081,2821,https://sallysbakingaddiction.com/soft-chewy-oatmeal-raisin-cookies/,"Unsalted Butter, Brown Sugar, Granulated Sugar, Eggs, Pure Vanilla Extract, Molasses, All-Purpose Flour, Baking Soda, Ground Cinnamon, Salt, Rolled Oats, Raisins, Walnuts"
9,Soft Dinner Rolls,4.8,1064,2865,https://sallysbakingaddiction.com/soft-dinner-rolls/,"Milk, Yeast, Granulated Sugar, Eggs, Unsalted Butter, Salt, All-Purpose Flour, Bread Flour"


In [256]:
# --- Create Yield Column! ---

for i in top10.index:
    url = top10.loc[i, 'URL']
    page = session.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    body = soup.find(class_ = 'tasty-recipes-yield')
    top10.loc[i, 'Yields'] = str(body.text.strip()).title()

# ~ Quick Rearranging of Columns ~
top10 = top10[['Title', 'Rating', 'Rating Count', 'Comment Count', 'Yields', 'Ingredients', 'URL']]

In [258]:
# --- Create Data Frame for Ingredient Quantity! ---
top10

,Title,Rating,Rating Count,Comment Count,Yields,Ingredients,URL
0,Banana Muffins,4.9,2168,4061,10-12 Muffins,"All-Purpose Flour, Baking Powder, Baking Soda, Salt, Ground Cinnamon, Ground Nutmeg, Mashed Bananas, Unsalted Butter, Brown Sugar, Eggs, Pure Vanilla Extract, Milk",https://sallysbakingaddiction.com/banana-muffins/
1,Chewy Chocolate Chip Cookies,4.7,1952,6344,16 Xl Cookies Or 20 Medium/Large Cookies,"All-Purpose Flour, Baking Soda, Cornstarch, Salt, Unsalted Butter, Brown Sugar, Granulated Sugar, Eggs, Pure Vanilla Extract, Semi-Sweet Chocolate Chips",https://sallysbakingaddiction.com/chewy-chocolate-chip-cookies/
2,Best Banana Bread Recipe,4.8,1756,4164,1 Loaf,"All-Purpose Flour, Baking Soda, Salt, Ground Cinnamon, Unsalted Butter, Brown Sugar, Eggs, Mashed Bananas, Plain Greek Yogurt, Sour Cream, Pure Vanilla Extract",https://sallysbakingaddiction.com/best-banana-bread-recipe/
3,Whole Wheat Bread,4.9,1589,3038,1 Loaf,"Water, Milk, Yeast, Wheat Flour, Honey, Unsalted Butter, Lemon Juice, Salt",https://sallysbakingaddiction.com/whole-wheat-bread/
4,Triple Chocolate Layer Cake,4.9,1370,4954,Serves 12-16,"All-Purpose Flour, Cocoa Powder, Granulated Sugar, Baking Soda, Baking Powder, Salt, Espresso Powder, Vegetable Oil, Eggs, Pure Vanilla Extract, Buttermilk, Hot Coffee, Unsalted Butter, Confectioners Sugar, Unsweetened Cocoa Powder, Heavy Cream",https://sallysbakingaddiction.com/triple-chocolate-layer-cake/
5,Lemon Bars Recipe,4.7,1298,3158,24 Bars,"Unsalted Butter, Granulated Sugar, Pure Vanilla Extract, Salt, All-Purpose Flour, Eggs, Lemon Juice, Lemon Zest",https://sallysbakingaddiction.com/lemon-bars-recipe/
6,Sandwich Bread,4.9,1144,3392,1 Loaf,"Water, Milk, Yeast, Granulated Sugar, Unsalted Butter, Salt, All-Purpose Flour, Bread Flour",https://sallysbakingaddiction.com/sandwich-bread/
7,Homemade Artisan Bread,4.8,1137,3219,2 8-Inch Loaves,"Bread Flour, Yeast, Salt, Water",https://sallysbakingaddiction.com/homemade-artisan-bread/
8,Soft Chewy Oatmeal Raisin Cookies,4.8,1081,2821,26-30 Cookies,"Unsalted Butter, Brown Sugar, Granulated Sugar, Eggs, Pure Vanilla Extract, Molasses, All-Purpose Flour, Baking Soda, Ground Cinnamon, Salt, Rolled Oats, Raisins, Walnuts",https://sallysbakingaddiction.com/soft-chewy-oatmeal-raisin-cookies/
9,Soft Dinner Rolls,4.8,1064,2865,14-16 Rolls,"Milk, Yeast, Granulated Sugar, Eggs, Unsalted Butter, Salt, All-Purpose Flour, Bread Flour",https://sallysbakingaddiction.com/soft-dinner-rolls/
